# score_runs_lora_val — canonical scorer, per checkpoint
The scorer from `score_runs.ipynb`, unchanged: same break patch, same `resize_ref`, same `score_row` re-indented into the per-step loop, same `max_workers=8`. The only wrapping is an outer loop over steps 1000/2000/3000/4000, writing `val__lora_r{RANK}_step{N}.jsonl` per step. Rate limits crash a pass non-destructively — rows flush per row, the done-set skips them, error rows stay retryable — so re-run the scoring cell until each step prints `to score 0`, exactly as with every other arm. One notebook per rank; run three in parallel.

## 1. Repo, scorer patch

In [ ]:
import os, subprocess, sys
REPO = "/content/OmniGen2"
if not os.path.isdir(REPO):
    subprocess.run(["git","clone","https://github.com/VectorSpaceLab/OmniGen2.git",REPO], check=True)
subprocess.run([sys.executable,"-m","pip","install","-q","regex"], check=True)   # json_util imports `regex`
sys.path.insert(0, REPO)
os.chdir(REPO)

# without the break, evaluate() re-runs a successful parse to max_tries: 6 calls per row instead of 2
p = "/content/OmniGen2/omnicontext/omnicontext_score.py"
lines = open(p).read().split("\n")
if not any("added: stop after successful parse" in l for l in lines):
    out = []
    for ln in lines:
        out.append(ln)
        if 'raise Exception("rate_limit_exceeded")' in ln:
            out.append("                break   # added: stop after successful parse")
    open(p, "w").write("\n".join(out))
    print("patched")
else:
    print("already patched")

patched


## 2. Drive, run config

In [ ]:
from google.colab import drive, userdata
import os, json
drive.mount("/content/drive")
os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")

DRIVE    = "/content/drive/MyDrive/Colab Notebooks/My Master's Thesis/Data"
DATASET  = "val"             # 98-row held-out slice for step selection
RANK     = 8                 # 4 | 8 | 16 — set per notebook
STEPS    = [1000, 2000, 3000, 4000]
RUN      = f"lora_r{RANK}"

for step in STEPS:
    gen = f"{DRIVE}/gen/{DATASET}/{RUN}__step{step}"
    n = len([f for f in os.listdir(gen) if f.endswith(".png")]) if os.path.isdir(gen) else 0
    print(f"r{RANK} step {step}: {n} images at {gen}")

Mounted at /content/drive
r8 step 1000: 98 images at /content/drive/MyDrive/Colab Notebooks/My Master's Thesis/Data/gen/val/lora_r8__step1000
r8 step 2000: 98 images at /content/drive/MyDrive/Colab Notebooks/My Master's Thesis/Data/gen/val/lora_r8__step2000
r8 step 3000: 98 images at /content/drive/MyDrive/Colab Notebooks/My Master's Thesis/Data/gen/val/lora_r8__step3000
r8 step 4000: 98 images at /content/drive/MyDrive/Colab Notebooks/My Master's Thesis/Data/gen/val/lora_r8__step4000


## 3. Reference images

In [ ]:
IMGS, EXPECTED = "/content/imgs", 37063
TAR = f"{DRIVE}/imgs_subset.tar"

def n_files(d): return sum(len(f) for _, _, f in os.walk(d)) if os.path.isdir(d) else 0

if n_files(IMGS) < EXPECTED:
    os.makedirs(IMGS, exist_ok=True)
    r = subprocess.run(f'cp "{TAR}" /content/imgs_subset.tar && tar -xf /content/imgs_subset.tar -C {IMGS}',
                       shell=True, capture_output=True, text=True)
    if r.returncode: raise RuntimeError(f"rc={r.returncode}\n{r.stderr[:800]}")
    print("restored", n_files(IMGS), "/", EXPECTED)
    assert n_files(IMGS) >= EXPECTED, "restore incomplete"
else:
    print("images complete:", n_files(IMGS))

for s in ["train", "val", "test", "screen", "screen_small"]:
    src = f"{DRIVE}/{s}.jsonl"
    if os.path.exists(src) and not os.path.exists(f"/content/{s}.jsonl"):
        subprocess.run(["cp", src, f"/content/{s}.jsonl"])

rows = [json.loads(l) for l in open(f"/content/{DATASET}.jsonl", encoding="utf-8")]
# indices generated across all step dirs (each step scored separately in the loop below)
all_have = sorted({int(f[:-4])
                   for step in STEPS
                   for f in os.listdir(f"{DRIVE}/gen/{DATASET}/{RUN}__step{step}")
                   if os.path.isdir(f"{DRIVE}/gen/{DATASET}/{RUN}__step{step}") and f.endswith(".png") and f[:-4].isdigit()})
assert all_have, f"no generated images found for {RUN} across steps {STEPS}"
assert max(all_have) < len(rows), f"image index {max(all_have)} outside {DATASET} ({len(rows)} rows)"
missing = [p for i in all_have for p in rows[i]["input_images"] if not os.path.exists(os.path.join(IMGS, p))]
assert not missing, f"{len(missing)} reference images missing, first: {missing[:3]}"
print(f"{DATASET}: {len(rows)} rows | {len(all_have)} generated indices across {len(STEPS)} steps")

images complete: 37063
val: 98 rows | 98 generated indices across 4 steps


## 4. Sanity — 3 rows

In [ ]:
import math
from PIL import Image
from omnicontext.omnicontext_score import OmniContextScore

scorer = OmniContextScore("https://api.openai.com/v1/chat/completions", userdata.get("OPENAI_API_KEY"))

def resize_ref(im):   # OmniContext default: cap at 1024^2, round to /16 — references only
    w, h = im.size
    cap = 1024*1024
    if w*h > cap:
        r = math.sqrt(cap/(w*h)); w, h = int(w*r), int(h*r)
    return im.resize((max(16, w//16*16), max(16, h//16*16)))

# 3-row smoke on the first step's images to confirm the scorer + adapter path before the full loop
_smoke_step = STEPS[0]
_smoke_GEN  = f"{DRIVE}/gen/{DATASET}/{RUN}__step{_smoke_step}"
_smoke_have = sorted(int(f[:-4]) for f in os.listdir(_smoke_GEN) if f.endswith(".png") and f[:-4].isdigit())
for i in _smoke_have[:3]:
    r = rows[i]
    refs = [resize_ref(Image.open(os.path.join(IMGS, p)).convert("RGB")) for p in r["input_images"]]
    gen  = Image.open(f"{_smoke_GEN}/{i:05d}.png").convert("RGB")
    res  = scorer.evaluate(refs + [gen], r["instruction"], with_scene=False)
    print(f"step {_smoke_step} row {i}: PF={res['PF_scores']['score']}  SC={res['SC_scores']['score']}")
    print(f"   PF: {str(res['PF_scores']['reasoning'])[:140]}")
    print(f"   SC: {str(res['SC_scores']['reasoning'])[:140]}\n")

step 1000 row 0: PF=8  SC=9
   PF: The second image best fulfills the instruction: the man stands confidently next to a sleek, open laptop displaying a vibrant presentation, a
   SC: The identity of the man in the final image is highly consistent with the original, with matching facial features, hairstyle, skin tone, and 

step 1000 row 1: PF=9  SC=9
   PF: The final image effectively follows the editing instruction: a professional man stands confidently next to a laptop that matches the one fro
   SC: The laptop in the final image closely matches the one from the reference image in terms of screen color gradient, aspect ratio, and frame st

step 1000 row 2: PF=8  SC=8
   PF: The final image meets most requirements: the man stands confidently with a bright smile, wearing a tailored dark suit, crisp white shirt, an
   SC: The man's facial features, hairstyle, and suit are highly consistent with those in image 1, including the dark tailored suit and white shirt



## 5. Full pass per checkpoint (resumable)

In [ ]:
import time
from concurrent.futures import ThreadPoolExecutor, as_completed

def num(x):           # fix_json can repair a bare score into [8]
    if isinstance(x, list) and x: x = x[0]
    return x if isinstance(x, (int, float)) else None

# identical scorer to score_runs.ipynb; only wrapped to run each step's directory in turn.
# rate limits crash a pass non-destructively: rows are flushed per row, the done-set skips them,
# error rows stay retryable -> just re-run this cell until every step prints "to score: 0".
for step in STEPS:
    GEN    = f"{DRIVE}/gen/{DATASET}/{RUN}__step{step}"
    SCORES = f"{DRIVE}/scores/{DATASET}__{RUN}_step{step}.jsonl"
    have   = sorted(int(f[:-4]) for f in os.listdir(GEN) if f.endswith(".png") and f[:-4].isdigit())

    done = set()
    if os.path.exists(SCORES):
        for l in open(SCORES, encoding="utf-8"):
            try:
                d = json.loads(l)
                if "error" not in d: done.add(d["i"])   # failures stay retryable
            except: pass

    def score_row(i):
        r  = rows[i]
        gp = f"{GEN}/{i:05d}.png"
        if not os.path.exists(gp): return {"i": i, "error": "missing_generated"}
        refs = [resize_ref(Image.open(os.path.join(IMGS, p)).convert("RGB")) for p in r["input_images"]]
        gen  = Image.open(gp).convert("RGB")            # official: generated image not resized
        time.sleep(0.2)
        res = scorer.evaluate(refs + [gen], r["instruction"], with_scene=False)   # original instruction, never the augmented one
        pf, sc = num(res["PF_scores"]["score"]), num(res["SC_scores"]["score"])
        if pf is None or sc is None: return {"i": i, "error": "unparsable_score"}
        # evaluate() substitutes random.randint(0,10) after 3 failed parses; that is not a measurement
        if any(str(res[k]["reasoning"]).startswith("guess_if_cannot_parse") for k in ("PF_scores", "SC_scores")):
            return {"i": i, "error": "guessed_score"}
        ups = ["_up" in os.path.basename(p) for p in r["input_images"]]
        return {"i": i, "uid": r["output_image"] + "|" + "|".join(sorted(r["input_images"])), "n_refs": len(r["input_images"]),
                "ref_kind": "all_up" if all(ups) else ("no_up" if not any(ups) else "mixed"),
                "PF": pf, "SC": sc,
                "PF_reason": res["PF_scores"]["reasoning"], "SC_reason": res["SC_scores"]["reasoning"]}

    todo = [i for i in have if i not in done]
    print(f"\nstep {step}: to score {len(todo)} (done: {len(done)}) -> {SCORES}")
    with open(SCORES, "a", encoding="utf-8") as f, ThreadPoolExecutor(max_workers=8) as ex:
        futs = [ex.submit(score_row, i) for i in todo]
        for n, fut in enumerate(as_completed(futs), 1):
            f.write(json.dumps(fut.result(), ensure_ascii=False) + "\n"); f.flush()
            if n % 25 == 0: print(f"  {n}/{len(todo)}")
    print(f"step {step} pass done -> {SCORES}")
print("\nall steps: re-run this cell until each prints 'to score 0'")


step 1000: to score 0 (done: 98) -> /content/drive/MyDrive/Colab Notebooks/My Master's Thesis/Data/scores/val__lora_r8_step1000.jsonl
step 1000 pass done -> /content/drive/MyDrive/Colab Notebooks/My Master's Thesis/Data/scores/val__lora_r8_step1000.jsonl

step 2000: to score 0 (done: 98) -> /content/drive/MyDrive/Colab Notebooks/My Master's Thesis/Data/scores/val__lora_r8_step2000.jsonl
step 2000 pass done -> /content/drive/MyDrive/Colab Notebooks/My Master's Thesis/Data/scores/val__lora_r8_step2000.jsonl

step 3000: to score 88 (done: 10) -> /content/drive/MyDrive/Colab Notebooks/My Master's Thesis/Data/scores/val__lora_r8_step3000.jsonl
  25/88
  50/88
  75/88
step 3000 pass done -> /content/drive/MyDrive/Colab Notebooks/My Master's Thesis/Data/scores/val__lora_r8_step3000.jsonl

step 4000: to score 98 (done: 0) -> /content/drive/MyDrive/Colab Notebooks/My Master's Thesis/Data/scores/val__lora_r8_step4000.jsonl
  25/98
  50/98
  75/98
step 4000 pass done -> /content/drive/MyDrive/Co

## 6. Step-selection curve

In [ ]:
import math, json
import numpy as np
ov = lambda r: math.sqrt(max(r["PF"],0)*max(r["SC"],0))

print(f"rank {RANK}: step-selection curve on the {DATASET} slice\n")
print(f"{'step':>6s} {'n':>4s} {'PF':>7s} {'SC':>7s} {'Overall':>8s}")
best = None
for step in STEPS:
    p = f"{DRIVE}/scores/{DATASET}__{RUN}_step{step}.jsonl"
    rs = [json.loads(l) for l in open(p, encoding="utf-8") if "error" not in json.loads(l)]
    sc = np.mean([r["SC"] for r in rs])
    print(f"{step:>6d} {len(rs):>4d} {np.mean([r['PF'] for r in rs]):>7.3f} {sc:>7.3f} {np.mean([ov(r) for r in rs]):>8.3f}")
    if best is None or sc > best[1]: best = (step, sc)
print(f"\nbest step by SC: {best[0]} (SC {best[1]:.3f})")
print("confirm this step against baseline on the full 488 test set before reporting.")

rank 8: step-selection curve on the val slice

  step    n      PF      SC  Overall
  1000   98   7.980   8.031    7.943
  2000   98   8.153   7.602    7.744
  3000   98   7.959   7.500    7.600
  4000   98   8.041   7.327    7.574

best step by SC: 1000 (SC 8.031)
confirm this step against baseline on the full 488 test set before reporting.
